# 🔧 Patch Cells — Drop These Into Your Existing Notebook

**Instructions:** Copy-paste each cell below into the correct location in your existing `models_2020_.ipynb`. Each patch is **standalone** — it uses variables already in your notebook's memory (like `df`, `logit_m`, `ord_result`, etc.) so you **do NOT** need to re-run the entire notebook.

---


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical models
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Scipy for chi-square tests
from scipy import stats

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully.")

All libraries loaded successfully.


In [2]:
# 0.2  Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('All libraries imported successfully.')

All libraries imported successfully.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import table
import io
import dataframe_image as dfi
import seaborn as sns

In [4]:
from pathlib import Path

In [13]:
# Define paths
DATA_PATH = Path("../data/no_missing")

In [14]:
# Load data
df_raw = pd.read_excel(DATA_PATH / "complete_birth_2020.xlsx")

In [15]:
# 1.2  Load the file

print(f'File loaded successfully.')
print(f'  Rows    : {len(df_raw):,}')
print(f'  Columns : {df_raw.shape[1]}')
print(f'\nColumn names:')
for c in df_raw.columns:
    print(f'  {c}')

File loaded successfully.
  Rows    : 261,550
  Columns : 18

Column names:
  Registered_Year
  Registered_Month
  Registered_District
  Birh_Year
  Birh_Year 2.0
  Birth_Month.0
  Birth_Month
  Gender
  Hospital or Not
  Multiple_Birth_Status
  Birth_Weight(grams)
  Birth_Order2.0
  Birth_Order
  Age of Mother
  Marital_Status
  District_of_Mother
  Race_of_Mother
  Race_of_Father


In [16]:
# 1.3  First look at the data
df_raw.head(5)

,Registered_Year,Registered_Month,Registered_District,Birh_Year,Birh_Year 2.0,Birth_Month.0,Birth_Month,Gender,Hospital or Not,Multiple_Birth_Status,Birth_Weight(grams),Birth_Order2.0,Birth_Order,Age of Mother,Marital_Status,District_of_Mother,Race_of_Mother,Race_of_Father
0,2020,January,Colombo,2019,2019,11,November,Male,Hospital,Singleton,3050,2,Second,31,Married,Gampaha,Sinhalese,Sinhalese
1,2020,January,Colombo,2019,2019,12,December,Female,Hospital,Singleton,3320,1,First,26,Married,Gampaha,Sinhalese,Sinhalese
2,2020,January,Colombo,2019,2019,12,December,Male,Hospital,Singleton,3940,3,Third,24,Married,Colombo,Srilankan Moor,Srilankan Moor
3,2020,January,Colombo,2019,2019,12,December,Female,Hospital,Singleton,2860,2,Second,28,Married,Colombo,Srilankan Moor,Srilankan Moor
4,2020,January,Colombo,2019,2019,10,October,Male,Hospital,Singleton,2850,2,Second,31,Married,Puttalam,Sinhalese,Sinhalese


In [17]:
# 2.1  Work on a clean copy and standardise column names
df = df_raw.copy()

In [18]:
# 2.2  Rename columns to short working names
df = df.rename(columns={
    'Birth_Order'          : 'parity',
    'Age of Mother'        : 'maternal_age',
    'Marital_Status'       : 'marital_status',
    'Race_of_Mother'       : 'race_mother',
    'Race_of_Father'       : 'race_father',
    'Gender'               : 'Gender ',
    'Hospital_or_Not'      : 'place_delivery',
    'Multiple_Birth_Status': 'multiple_birth',
    'Birth_Weightgrams'    : 'birth_weight_g',
    'Registered_District'  : 'reg_district',
    'Registered_Year'      : 'reg_year',
    'Registered_Month'     : 'reg_month',
    'Birh_Year'            : 'birth_year',
    'Birth_Month'          : 'birth_month',
    'District_of_Mother'   : 'district_mother',
})

print('Working column names:')
print(df.columns.tolist())

Working column names:
['reg_year', 'reg_month', 'reg_district', 'birth_year', 'Birh_Year 2.0', 'Birth_Month.0', 'birth_month', 'Gender ', 'Hospital or Not', 'multiple_birth', 'Birth_Weight(grams)', 'Birth_Order2.0', 'parity', 'maternal_age', 'marital_status', 'district_mother', 'race_mother', 'race_father']


In [19]:
# 2.3  Check unique Birth_Order text values before mapping
print('Unique values in Birth_Order column:')
print(df['parity'].value_counts(dropna=False))

Unique values in Birth_Order column:
parity
First      106665
Second      96224
Third       47648
Fourth       8942
Fifth        1679
Sixth         297
Seventh        71
Eighth         19
Nineth          5
Name: count, dtype: int64


In [20]:
# 2.4  Map text parity labels to numeric 1–9
BIRTH_ORDER_MAP = {
    'First'  : 1,
    'Second' : 2,
    'Third'  : 3,
    'Fourth' : 4,
    'Fifth'  : 5,
    'Sixth'  : 6,
    'Seventh': 7,
    'Eighth' : 8,
    'Nineth' : 9,   # keeping original spelling from your data
}

# Strip accidental spaces then map
df['parity_text'] = df['parity'].astype(str).str.strip()  # save text version
df['parity'] = df['parity_text'].map(BIRTH_ORDER_MAP)     # overwrite with numeric

# Warn about any unmapped values
unmapped = df[df['parity'].isna()]['parity_text'].unique()
if len(unmapped) > 0:
    print(f'WARNING: Unmapped values found. Add these to BIRTH_ORDER_MAP: {unmapped}')
else:
    print('All parity labels mapped successfully.')

print('\nNumeric parity distribution:')
print(df['parity'].value_counts().sort_index())


All parity labels mapped successfully.

Numeric parity distribution:
parity
1    106665
2     96224
3     47648
4      8942
5      1679
6       297
7        71
8        19
9         5
Name: count, dtype: int64


In [21]:
# 2.5  Create binary outcome for Model 3
# 1 = First-born  |  0 = Later-born (parity 2 to 9)
df['parity_binary'] = (df['parity'] == 1).astype(int)

print(f'First-born  (parity = 1) : {(df["parity_binary"]==1).sum():,}')
print(f'Later-born  (parity >= 2): {(df["parity_binary"]==0).sum():,}')
print(f'First-birth percentage   : {df["parity_binary"].mean()*100:.1f}%')

First-born  (parity = 1) : 106,665
Later-born  (parity >= 2): 154,885
First-birth percentage   : 40.8%


## PATCH 1: Collapse Parity 1–9 → 1, 2, 3, 4+
📌 **Insert AFTER** cell 2.5 (where `parity_binary` is created)

**Comment:** *"do we have to go 1 to 9? can we create 1, 2, 3, and 4<. this is still ordinal?"*


In [22]:
# ============================================================
# PATCH 1: Collapse parity → 4 categories (1, 2, 3, 4+)
# Supervisor: "can we create 1, 2, 3, and 4<"
# INSERT AFTER cell 2.5
# ============================================================

df['parity_collapsed'] = df['parity'].apply(
    lambda x: min(x, 4) if pd.notna(x) else np.nan
).astype('Int64')

print('COLLAPSED PARITY DISTRIBUTION:')
print(df['parity_collapsed'].value_counts().sort_index())
print(f'\nCategories: 1=First, 2=Second, 3=Third, 4=Fourth-or-higher')
print(f'Total: {df["parity_collapsed"].notna().sum():,}')


COLLAPSED PARITY DISTRIBUTION:
parity_collapsed
1    106665
2     96224
3     47648
4     11013
Name: count, dtype: Int64

Categories: 1=First, 2=Second, 3=Third, 4=Fourth-or-higher
Total: 261,550


## PATCH 2: Collapse Multiple Birth → Binary (Singleton vs Multiple)
📌 **Insert AFTER** cell 2.8 (standardize categoricals)

**Comment:** *"if too high have single births and multiple births only; as a binary?"*


In [23]:
# ============================================================
# PATCH 2: Collapse multiple_birth → binary
# Supervisor: "have single births and multiple births only; as a binary"
# INSERT AFTER cell 2.8
# ============================================================

df['is_multiple'] = (df['multiple_birth'].astype(str).str.strip().str.title() != 'Singleton').astype(int)

print('Multiple birth (binary):')
print(df['is_multiple'].value_counts().rename({0: 'Singleton', 1: 'Multiple'}))
print(f'\nMultiple births: {df["is_multiple"].sum():,} ({df["is_multiple"].mean()*100:.2f}%)')


Multiple birth (binary):
is_multiple
Singleton    258786
Multiple       2764
Name: count, dtype: int64

Multiple births: 2,764 (1.06%)


## PATCH 3: Add Province Variable
📌 **Insert AFTER** Patch 2

**Comment:** *"can we include the province or a similar indicator for residence?"*


In [24]:
# ============================================================
# PATCH 3: Create Province from district_mother
# Supervisor: "can we include the province or a similar indicator for residence?"
# INSERT AFTER Patch 2
# ============================================================

DISTRICT_TO_PROVINCE = {
    'Colombo': 'Western', 'Gampaha': 'Western', 'Kalutara': 'Western',
    'Kandy': 'Central', 'Matale': 'Central', 'Nuwara Eliya': 'Central',
    'Galle': 'Southern', 'Matara': 'Southern', 'Hambantota': 'Southern',
    'Jaffna': 'Northern', 'Kilinochchi': 'Northern', 'Mannar': 'Northern',
    'Mullaitivu': 'Northern', 'Vavuniya': 'Northern',
    'Batticaloa': 'Eastern', 'Ampara': 'Eastern', 'Trincomalee': 'Eastern',
    'Kurunegala': 'North Western', 'Puttalam': 'North Western',
    'Anuradhapura': 'North Central', 'Polonnaruwa': 'North Central',
    'Badulla': 'Uva', 'Monaragala': 'Uva',
    'Ratnapura': 'Sabaragamuwa', 'Kegalle': 'Sabaragamuwa',
}

df['district_clean'] = df['district_mother'].astype(str).str.strip().str.title()
df['province'] = df['district_clean'].map(DISTRICT_TO_PROVINCE)

print('Province distribution:')
print(df['province'].value_counts().sort_index())
print(f'\nMissing province: {df["province"].isna().sum():,}')
# Drop rows with missing province
df = df.dropna(subset=['province']).reset_index(drop=True)
print(f'Rows after province filter: {len(df):,}')


Province distribution:
province
Central          37816
Eastern          15516
North Central    18644
North Western    11197
Northern         15464
Sabaragamuwa     25585
Southern         35710
Uva              12954
Western          58120
Name: count, dtype: int64

Missing province: 30,544
Rows after province filter: 231,006


## PATCH 4: Replace VIF with GVIF
📌 **Replace** the existing VIF cell entirely

**Comment:** *"VIF should be checked only for quantitative variables (for categorical variables it is meaningless). use GVIF"*


In [26]:
# ============================================================
# PATCH 4: GVIF (Generalized VIF) — replaces VIF
# Supervisor: "VIF should be checked only for quantitative variables. use GVIF"
# Fox & Monette (1992), JASA 87(417), 178-183
# ============================================================

import pandas as pd
import numpy as np

def compute_gvif(df_input, predictors):
    """
    Compute GVIF and GVIF^(1/(2*df)) for categorical + continuous predictors.
    GVIF^(1/(2*df)) > sqrt(10) ≈ 3.16  ↔  VIF > 10 (problematic)
    """
    # First, check which predictors exist in the dataframe
    available_predictors = [v for v in predictors if v in df_input.columns]
    missing_predictors = [v for v in predictors if v not in df_input.columns]
    
    if missing_predictors:
        print(f"WARNING: These predictors are missing from the dataframe: {missing_predictors}")
        print(f"Available columns: {list(df_input.columns)}")
        if not available_predictors:
            raise ValueError("No valid predictors found!")
    
    if len(available_predictors) < 2:
        raise ValueError(f"Need at least 2 predictors, but only found {len(available_predictors)}")
    
    # Identify categorical and continuous variables
    cat_vars = []
    cont_vars = []
    
    for v in available_predictors:
        dtype = df_input[v].dtype
        if dtype == 'object' or dtype.name == 'category' or dtype == 'bool':
            cat_vars.append(v)
        else:
            cont_vars.append(v)
    
    print(f"\nCategorical variables: {cat_vars}")
    print(f"Continuous variables: {cont_vars}")
    
    # Create dummy variables for categorical predictors
    X = pd.get_dummies(df_input[available_predictors], drop_first=True, dtype=float)
    
    # Map original variables to their dummy columns
    var_cols = {}
    for var in cont_vars:
        var_cols[var] = [var]
    for var in cat_vars:
        var_cols[var] = [c for c in X.columns if c == var or (c.startswith(var + '_') and c != var)]
    
    # Remove any empty entries
    var_cols = {k: v for k, v in var_cols.items() if v}
    
    if len(var_cols) < 2:
        raise ValueError(f"After processing, only {len(var_cols)} variables remain. Need at least 2.")
    
    # Compute correlation matrix
    R = X.corr().values
    try:
        R_inv = np.linalg.inv(R)
    except np.linalg.LinAlgError:
        print("Note: Using pseudo-inverse due to singular correlation matrix")
        R_inv = np.linalg.pinv(R)
    
    col_list = list(X.columns)
    results = []
    
    for var, cols in var_cols.items():
        d = len(cols)
        try:
            idx = [col_list.index(c) for c in cols]
            R_ii = R_inv[np.ix_(idx, idx)]
            R_orig_ii = R[np.ix_(idx, idx)]
            
            # Compute GVIF
            det_R_ii = np.linalg.det(R_ii)
            det_R_orig_ii = np.linalg.det(R_orig_ii)
            gvif = det_R_ii * det_R_orig_ii
            
            # Ensure GVIF is not negative due to numerical issues
            if gvif < 0:
                gvif = max(gvif, 0)
            
            gvif_adj = gvif ** (1 / (2 * d)) if gvif > 0 else 0
            
            results.append({
                'Variable': var,
                'GVIF': round(gvif, 4),
                'df': d,
                'GVIF^(1/(2*df))': round(gvif_adj, 4),
                'Equiv_VIF': round(gvif_adj**2, 4),
                'Status': '⚠ HIGH' if gvif_adj > 3.16 else '✓ OK'
            })
        except Exception as e:
            print(f"Warning: Could not compute GVIF for {var}: {e}")
            results.append({
                'Variable': var,
                'GVIF': np.nan,
                'df': d,
                'GVIF^(1/(2*df))': np.nan,
                'Equiv_VIF': np.nan,
                'Status': '⚠ ERROR'
            })
    
    return pd.DataFrame(results)

# First, check what columns are available in your dataframe
print("Available columns in dataframe:")
print(list(df.columns))
print("\n" + "="*85 + "\n")

# Define predictors based on what's likely available in your dataset
# You'll need to adjust these based on your actual column names

# Common column names (adjust based on your actual data)
potential_predictors = ['age_group', 'marital_status', 'race_mother', 'Gender',
                        'place_delivery', 'province', 'Birth_Weight_grams',
                        'birth_weight', 'is_multiple']

# Check which ones actually exist
existing_predictors = [col for col in potential_predictors if col in df.columns]

print(f"Found predictors: {existing_predictors}")

# If you don't have age_group, create it from age variable if available
if 'age_group' not in df.columns and 'age' in df.columns:
    print("\nCreating age_group from age variable...")
    df['age_group'] = pd.cut(df['age'], 
                             bins=[0, 20, 25, 30, 35, 40, 100],
                             labels=['<20', '20-24', '25-29', '30-34', '35-39', '40+'])
    existing_predictors.append('age_group')

# Prepare the GVIF predictors list
gvif_predictors = [col for col in existing_predictors if col != 'is_multiple']

# Add is_multiple as categorical for GVIF if it exists
if 'is_multiple' in df.columns:
    df['is_multiple_cat'] = df['is_multiple'].map({0: 'Singleton', 1: 'Multiple'})
    gvif_predictors_full = gvif_predictors + ['is_multiple_cat']
else:
    print("Warning: 'is_multiple' column not found. Using only available predictors.")
    gvif_predictors_full = gvif_predictors

print(f"\nFinal predictors for GVIF analysis: {gvif_predictors_full}")
print("\n" + "="*85 + "\n")

# Compute GVIF
try:
    gvif_results = compute_gvif(df, gvif_predictors_full)
    
    print('=' * 85)
    print('GENERALIZED VIF (GVIF) — Fox & Monette (1992)')
    print('Threshold: GVIF^(1/(2*df)) > √10 ≈ 3.16 → problematic multicollinearity')
    print('=' * 85)
    print(gvif_results.to_string(index=False))
    
    # Save results
    gvif_results.to_excel('gvif_results_revised.xlsx', index=False)
    print('\n✓ Saved: gvif_results_revised.xlsx')
    
    # Summary of findings
    print("\n" + "="*85)
    print("SUMMARY:")
    high_gvif = gvif_results[gvif_results['GVIF^(1/(2*df))'] > 3.16]
    if len(high_gvif) > 0:
        print(f"⚠ HIGH multicollinearity detected for: {list(high_gvif['Variable'])}")
        print("Consider removing or combining these variables.")
    else:
        print("✓ No problematic multicollinearity detected.")
    print("="*85)
    
except Exception as e:
    print(f"\nERROR in GVIF computation: {e}")
    print("\nTroubleshooting tips:")
    print("1. Check that all predictor variables exist in the dataframe")
    print("2. Ensure numeric variables are actually numeric (not object dtype)")
    print("3. Check for missing values in predictor variables")
    print("4. Try with a subset of predictors to identify the problematic variable")

Available columns in dataframe:
['reg_year', 'reg_month', 'reg_district', 'birth_year', 'Birh_Year 2.0', 'Birth_Month.0', 'birth_month', 'Gender ', 'Hospital or Not', 'multiple_birth', 'Birth_Weight(grams)', 'Birth_Order2.0', 'parity', 'maternal_age', 'marital_status', 'district_mother', 'race_mother', 'race_father', 'parity_text', 'parity_binary', 'parity_collapsed', 'is_multiple', 'district_clean', 'province', 'is_multiple_cat']


Found predictors: ['marital_status', 'race_mother', 'province', 'is_multiple']

Final predictors for GVIF analysis: ['marital_status', 'race_mother', 'province', 'is_multiple_cat']



Categorical variables: []
Continuous variables: ['marital_status', 'race_mother', 'province', 'is_multiple_cat']
GENERALIZED VIF (GVIF) — Fox & Monette (1992)
Threshold: GVIF^(1/(2*df)) > √10 ≈ 3.16 → problematic multicollinearity
       Variable  GVIF  df  GVIF^(1/(2*df))  Equiv_VIF  Status
 marital_status   NaN   1              NaN        NaN ⚠ ERROR
    race_mother   NaN   

## PATCH 5: Re-encode Predictors (with province, binary multiple birth)
📌 **Insert BEFORE** the model fitting cells (before ordinal / count / binary models)

This updates `X_encoded` to include province and use binary multiple birth. The models will then use these updated predictors.


In [28]:
# ============================================================
# PATCH 5: Re-encode predictors with revised variables
# Uses: binary multiple birth, province, collapsed parity
# INSERT BEFORE model fitting cells
# ============================================================

cat_predictors = ['age_group', 'marital_status', 'race_mother', 'Gender ',
                  'place_delivery', 'is_multiple_cat', 'province']
cont_predictors = ['Birth_Weight(grams)']

X_encoded = pd.get_dummies(df[cat_predictors], drop_first=True, dtype=float)
for cp in cont_predictors:
    X_encoded[cp] = df[cp].values

print(f'Updated design matrix: {X_encoded.shape}')
print(f'Predictor columns ({len(X_encoded.columns)}):')
for c in X_encoded.columns:
    print(f'  {c}')


KeyError: "['age_group', 'place_delivery'] not in index"

## PATCH 6: Updated Chi-Square Tests (with province, collapsed parity)
📌 **Replace** existing chi-square cell


In [ ]:
# ============================================================
# PATCH 6: Chi-Square tests — revised with province + collapsed parity
# REPLACE existing chi-square cell
# ============================================================

chi2_vars = ['age_group', 'marital_status', 'race_mother', 'Gender ',
             'place_delivery', 'is_multiple_cat', 'province']

chi2_results = []
for var in chi2_vars:
    ct = pd.crosstab(df[var], df['parity_collapsed'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    sig = 'Yes ***' if p < 0.001 else ('Yes **' if p < 0.01 else ('Yes *' if p < 0.05 else 'No'))
    chi2_results.append({
        'Variable': var, 'Chi2': round(chi2, 2),
        'df': dof, 'p-value': round(p, 6), 'Significant': sig
    })

chi2_df = pd.DataFrame(chi2_results)
print('CHI-SQUARE TESTS (outcome = collapsed parity 1/2/3/4+):')
print(chi2_df.to_string(index=False))
chi2_df.to_excel('results_chisquare_revised.xlsx', index=False)
print('\nSaved: results_chisquare_revised.xlsx')


## PATCH 7: Re-fit Ordinal Logistic with Collapsed Parity (1/2/3/4+)
📌 **Replace** the ordinal model fitting cell

⏱ This cell DOES require re-fitting, but it's much faster with 4 categories instead of 9 (3 thresholds instead of 8).


In [ ]:
# ============================================================
# PATCH 7: Re-fit ordinal model with collapsed parity
# MUCH FASTER: 4 categories (3 thresholds) vs 9 (8 thresholds)
# REPLACE the ordinal model fitting cell
# ============================================================

y_ord = df['parity_collapsed'].values

ord_model = OrderedModel(y_ord, X_encoded, distr='logit')
ord_result = ord_model.fit(method='bfgs', maxiter=5000, disp=False)

print('=' * 60)
print('MODEL 1 — ORDINAL LOGISTIC (COLLAPSED: 1/2/3/4+)')
print('=' * 60)
print(ord_result.summary())

# Fit statistics
print(f'\nLog-Likelihood : {ord_result.llf:.4f}')
print(f'AIC            : {ord_result.aic:.4f}')
print(f'BIC            : {ord_result.bic:.4f}')
print(f'Converged      : {ord_result.mle_retvals["converged"]}')


## PATCH 8: Re-fit Count Model with Updated Predictors
📌 **Replace** count model cell


In [ ]:
# ============================================================
# PATCH 8: Re-fit count model with updated predictors
# REPLACE count model fitting cell
# ============================================================

y_count = df['parity'].values
X_count_const = sm.add_constant(X_encoded)

poisson_m = sm.GLM(y_count, X_count_const, family=sm.families.Poisson()).fit()

try:
    nb_m = sm.GLM(y_count, X_count_const, family=sm.families.NegativeBinomial()).fit()
    best_count_m = nb_m if nb_m.aic < poisson_m.aic else poisson_m
    best_count_name = 'Negative Binomial' if nb_m.aic < poisson_m.aic else 'Poisson'
    print(f'Poisson AIC: {poisson_m.aic:.2f}  |  NB AIC: {nb_m.aic:.2f}')
except:
    best_count_m = poisson_m
    best_count_name = 'Poisson'

print(f'Selected: {best_count_name}')
print(f'Log-Likelihood: {best_count_m.llf:.4f}')
print(f'AIC: {best_count_m.aic:.4f}')
print(best_count_m.summary())


## PATCH 9: Re-fit Binary Logistic with Updated Predictors
📌 **Replace** binary logistic fitting cell


In [ ]:
# ============================================================
# PATCH 9: Re-fit binary logistic with updated predictors
# REPLACE binary logistic fitting cell
# ============================================================

y_bin = df['parity_binary'].values
X_const = sm.add_constant(X_encoded)

logit_m = sm.Logit(y_bin, X_const).fit(disp=False)

print('=' * 60)
print('MODEL 3 — BINARY LOGISTIC (UPDATED PREDICTORS)')
print('=' * 60)
print(logit_m.summary())


## PATCH 10: Add Classification Disclaimer
📌 **Insert BEFORE** the ROC / confusion matrix cell (cell 7.4 or 7.5)

**Comment:** *"are we going to predict also?? mention that our intention is not to predict but to identify the effects of predictors."*


In [ ]:
# ============================================================
# PATCH 10: Classification disclaimer
# Supervisor: "mention that our intention is not to predict 
#              but to identify the effects of predictors"
# INSERT BEFORE the ROC / confusion matrix cell
# ============================================================

print('=' * 70)
print('⚠  IMPORTANT NOTE ON CLASSIFICATION METRICS')
print('=' * 70)
print('''
The classification performance metrics (accuracy, sensitivity, specificity,
ROC curve, AUC, confusion matrix) reported below are presented as
SUPPLEMENTARY MODEL DIAGNOSTICS ONLY.

The PRIMARY OBJECTIVE of this study is to:
  → Identify and quantify the EFFECTS of predictor variables on birth parity

The study does NOT aim to:
  ✗ Build a predictive classification model
  ✗ Develop a screening tool for first-born identification

These metrics serve to validate the model's overall adequacy in capturing
the data structure and to assess goodness-of-fit, not for prediction.
''')
print('=' * 70)


## PATCH 11: Fix ŷ (hat) Notation in Fitted Equations
📌 **Replace** the model equation printing cells

**Comment:** *"hat is missing in the response. check all models"*


In [ ]:
# ============================================================
# PATCH 11: Fitted equations with ŷ hat notation
# Supervisor: "hat is missing in the response"
# REPLACE model equation cells
# ============================================================

print('=' * 75)
print('FITTED MODEL EQUATIONS (with ŷ / p̂ / μ̂ hat notation)')
print('=' * 75)

# --- Model 1: Ordinal ---
print('\n─── MODEL 1: ORDINAL LOGISTIC (Proportional Odds) ───')
print('logit[ P̂(Y ≤ j | x) ] = α̂ⱼ + Σ β̂ₖxₖ')
print()
params = ord_result.params
predictor_names = list(X_encoded.columns)
n_thresholds = len(df['parity_collapsed'].unique()) - 1

# Print significant predictors only
for i, name in enumerate(predictor_names):
    p = ord_result.pvalues[i]
    if p < 0.05:
        sign = '+' if params[i] >= 0 else '-'
        print(f'  {sign} {abs(params[i]):.4f} × {name}  (p={p:.4f})')

print(f'\n  Threshold parameters (α̂ⱼ):')
for j in range(n_thresholds):
    idx = len(predictor_names) + j
    print(f'    α̂_{j+1}/{j+2} = {params[idx]:.4f}')

# --- Model 2: Count ---
print(f'\n─── MODEL 2: {best_count_name.upper()} REGRESSION ───')
print('log(μ̂ᵢ) = β̂₀ + Σ β̂ₖxₖ')
print(f'  Intercept β̂₀ = {best_count_m.params[0]:.4f}')
for name in X_encoded.columns:
    if name in best_count_m.params.index and best_count_m.pvalues[name] < 0.05:
        coef = best_count_m.params[name]
        sign = '+' if coef >= 0 else '-'
        print(f'  {sign} {abs(coef):.4f} × {name}  (p={best_count_m.pvalues[name]:.4f})')

# --- Model 3: Binary ---
print('\n─── MODEL 3: BINARY LOGISTIC ───')
print('logit(p̂) = ln(p̂/(1-p̂)) = β̂₀ + Σ β̂ₖxₖ')
print(f'  Intercept β̂₀ = {logit_m.params[0]:.4f}')
for name in X_encoded.columns:
    if name in logit_m.params.index and logit_m.pvalues[name] < 0.05:
        coef = logit_m.params[name]
        sign = '+' if coef >= 0 else '-'
        print(f'  {sign} {abs(coef):.4f} × {name}  (p={logit_m.pvalues[name]:.4f})')
print(f'\n  p̂ = P̂(Y=1|x) = 1 / (1 + exp(−logit(p̂)))')
print('=' * 75)


## PATCH 12: Fix "values not fully seen" — Full Decimal OR Tables
📌 **Replace** the OR table printing cells

**Comment:** *"values are not fully seen"*


In [ ]:
# ============================================================
# PATCH 12: Full-precision OR/IRR tables
# Supervisor: "values are not fully seen"
# REPLACE OR table cells — shows 4 decimals for all values
# ============================================================

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 120)

# --- Ordinal ORs ---
ord_params = ord_result.params[:len(X_encoded.columns)]
ord_pvals  = ord_result.pvalues[:len(X_encoded.columns)]
ord_conf   = ord_result.conf_int().iloc[:len(X_encoded.columns)]

ord_or = pd.DataFrame({
    'OR':       np.exp(ord_params).round(4),
    'CI_lower': np.exp(ord_conf.iloc[:, 0]).round(4),
    'CI_upper': np.exp(ord_conf.iloc[:, 1]).round(4),
    'p_value':  ord_pvals.round(6),
    'Sig': [('***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))) for p in ord_pvals]
}, index=X_encoded.columns)

print('MODEL 1 — ORDINAL LOGISTIC: CUMULATIVE ODDS RATIOS (full precision)')
print(ord_or.to_string())
ord_or.to_excel('model1_ordinal_OR_revised.xlsx')
print('Saved: model1_ordinal_OR_revised.xlsx')

# --- Count IRRs ---
count_params = best_count_m.params[1:]
count_pvals  = best_count_m.pvalues[1:]
count_conf   = best_count_m.conf_int().iloc[1:]

count_irr = pd.DataFrame({
    'IRR':      np.exp(count_params).round(4),
    'CI_lower': np.exp(count_conf.iloc[:, 0]).round(4),
    'CI_upper': np.exp(count_conf.iloc[:, 1]).round(4),
    'p_value':  count_pvals.round(6),
    'Sig': [('***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))) for p in count_pvals]
}, index=X_encoded.columns)

print(f'\nMODEL 2 — {best_count_name.upper()}: IRRs (full precision)')
print(count_irr.to_string())
count_irr.to_excel('model2_count_IRR_revised.xlsx')
print(f'Saved: model2_count_IRR_revised.xlsx')

# --- Binary ORs ---
bin_params = logit_m.params[1:]
bin_pvals  = logit_m.pvalues[1:]
bin_conf   = logit_m.conf_int().iloc[1:]

binary_or = pd.DataFrame({
    'OR':       np.exp(bin_params).round(4),
    'CI_lower': np.exp(bin_conf.iloc[:, 0]).round(4),
    'CI_upper': np.exp(bin_conf.iloc[:, 1]).round(4),
    'p_value':  bin_pvals.round(6),
    'Sig': [('***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))) for p in bin_pvals]
}, index=X_encoded.columns)

print(f'\nMODEL 3 — BINARY LOGISTIC: ODDS RATIOS (full precision)')
print(binary_or.to_string())
binary_or.to_excel('model3_binary_OR_revised.xlsx')
print('Saved: model3_binary_OR_revised.xlsx')


## ✅ Summary — Order of Insertion

Run these patches in this order in your **existing** notebook:

| Step | Patch | Where to Insert | Re-fit needed? |
|------|-------|----------------|----------------|
| 1 | **Patch 1** — Collapse parity | After cell 2.5 | No (just adds column) |
| 2 | **Patch 2** — Binary multiple birth | After cell 2.8 | No (just adds column) |
| 3 | **Patch 3** — Add province | After Patch 2 | No (just adds column) |
| 4 | **Patch 4** — GVIF | Replace VIF cell | No (diagnostic only) |
| 5 | **Patch 5** — Re-encode predictors | Before models | No (creates X_encoded) |
| 6 | **Patch 6** — Chi-square tests | Replace chi-square cell | Fast (crosstabs) |
| 7 | **Patch 7** — Ordinal model | Replace ordinal cell | ⏱ Yes, but **much faster** (4 vs 9 categories) |
| 8 | **Patch 8** — Count model | Replace count cell | ⏱ Yes |
| 9 | **Patch 9** — Binary logistic | Replace binary cell | ⏱ Yes |
| 10 | **Patch 10** — Classification disclaimer | Before ROC/CM | No (just prints text) |
| 11 | **Patch 11** — ŷ hat equations | Replace equation cells | No (uses existing results) |
| 12 | **Patch 12** — Full-precision tables | Replace OR table cells | No (uses existing results) |

**Patches 1–6 and 10–12 are instant** (no model fitting).  
**Only Patches 7–9** require re-fitting, but the ordinal model will be **significantly faster** with 4 categories instead of 9.
